# Phase 2: Google OAuth Workshop

## What changed from Phase 1

In Phase 1 we built sessions and cookies with a simple email login. It worked, but anyone could type any email. We never verified identity.

In Phase 2 we replace the email form with Google login. **The session/cookie mechanism is identical.** The only change is HOW we confirm the user's email.

```
Phase 1:
  User types email → we trust it → create session → set cookie
                     ^^^^^^^^^^^
                     PROBLEM: no verification

Phase 2:
  User clicks "Login with Google" → Google verifies → we get email from Google → create session → set cookie
                                    ^^^^^^^^^^^^^^^^                              ^^^^^^^^^^^^^^^^^^^^^^^^
                                    GOOGLE verifies                               SAME as Phase 1
```

## Files changed

| File | What changed | What stayed the same |
|------|-------------|---------------------|
| `auth/routes.py` | Removed signup + login, added google_login + callback | `_create_session()`, logout, /me |
| `frontend/App.jsx` | Email form → "Continue with Google" button | Chat page, logout, cookie handling |
| `.env` | Added GOOGLE_CLIENT_ID and GOOGLE_CLIENT_SECRET | OPENAI_API_KEY |

# The Google Login Flow — 6 Steps

```
   Browser                      Your Server (FastAPI)                  Google
     |                                |                                  |
     |  1. Click "Login with Google"  |                                  |
     |------------------------------->|                                  |
     |                                |                                  |
     |  2. Redirect to Google         |                                  |
     |  Location: https://accounts.google.com/o/oauth2/v2/auth          |
     |  ?client_id=YOUR_ID            |                                  |
     |  &redirect_uri=localhost:8000/auth/callback                       |
     |  &scope=openid email profile   |                                  |
     |  &state=random_csrf_token      |                                  |
     |<-------------------------------|                                  |
     |                                                                   |
     |  3. User sees Google login page                                   |
     |  User enters their Google password                                |
     |  (YOUR APP NEVER SEES THE PASSWORD)                               |
     |------------------------------------------------------------------>|
     |                                                                   |
     |  4. Google redirects back to your server                          |
     |  Location: localhost:8000/auth/callback?code=ABCDEF&state=random  |
     |------------------------------->|                                  |
     |                                |                                  |
     |                                |  5. Exchange code for tokens     |
     |                                |  POST https://oauth2.googleapis.com/token
     |                                |  body: code + client_secret      |
     |                                |  (SERVER-TO-SERVER, secret safe) |
     |                                |--------------------------------->|
     |                                |                                  |
     |                                |  Google returns: access_token    |
     |                                |  Server calls: /userinfo         |
     |                                |  Gets: email, name, picture      |
     |                                |<---------------------------------|
     |                                |                                  |
     |  6. Server creates session     |                                  |
     |  (SAME _create_session()!)     |                                  |
     |  Set-Cookie: session_token=... |                                  |
     |  Redirect to frontend          |                                  |
     |<-------------------------------|                                  |
     |                                                                   |
     |  From here on: IDENTICAL to Phase 1                               |
     |  Cookie sent on every request, server looks up session, etc.      |
```

# Code Walkthrough: What Changed in auth/routes.py

## REMOVED (Phase 1 code)

These two endpoints are gone:
- `POST /auth/signup` — no more manual user creation
- `POST /auth/login` — no more trusting whatever email the user types

## ADDED: Step 1 — Redirect to Google

**File:** `auth/routes.py` → `google_login()`

```python
@router.get("/google/login")
def google_login():
    # Generate random state for CSRF protection
    state = secrets.token_hex(16)
    pending_states[state] = True

    # Build Google authorization URL
    oauth = OAuth2Session(
        client_id=GOOGLE_CLIENT_ID,
        redirect_uri=REDIRECT_URI,        # http://localhost:8000/auth/callback
        scope="openid email profile",     # what info we want
    )
    url, _ = oauth.create_authorization_url(GOOGLE_AUTH_URL, state=state)

    return RedirectResponse(url=url)      # browser goes to Google
```

**What this does:** When the user clicks "Login with Google", the frontend sends them to `/auth/google/login`. This endpoint builds a URL to Google's login page and redirects the browser there. The user sees Google's login form, NOT ours.

## ADDED: Steps 2-6 — Google Redirects Back

**File:** `auth/routes.py` → `google_callback()`

```python
@router.get("/callback")
def google_callback(code: str, state: str, db = Depends(get_db)):

    # Step 1: CSRF check
    if state not in pending_states:
        raise HTTPException(400, "Invalid state — possible CSRF attack")
    del pending_states[state]

    # Step 2: Exchange code for tokens (SERVER-TO-SERVER)
    oauth = OAuth2Session(
        client_id=GOOGLE_CLIENT_ID,
        client_secret=GOOGLE_CLIENT_SECRET,   # secret stays on server!
        redirect_uri=REDIRECT_URI,
    )
    oauth.fetch_token(GOOGLE_TOKEN_URL, code=code)

    # Step 3: Ask Google "who is this person?"
    user_info = oauth.get(GOOGLE_USERINFO_URL).json()
    email = user_info["email"]          # VERIFIED by Google
    name = user_info.get("name", email)
    picture = user_info.get("picture")

    # Step 4: Find or create user (Just-In-Time provisioning)
    user = db.query(User).filter(User.email == email).first()
    if not user:
        user = User(email=email, name=name, picture=picture)
        db.add(user)
        db.commit()
        db.refresh(user)

    # Step 5: Create session + set cookie — SAME AS PHASE 1!
    response = RedirectResponse(url=FRONTEND_URL, status_code=302)
    _create_session(user, response, db)

    return response
```

**The key insight:** Step 5 calls `_create_session()` — the exact same function from Phase 1. Same `secrets.token_hex(32)`, same INSERT INTO sessions, same `set_cookie(httponly=True)`. Nothing changed about sessions. We just replaced the "how do we know who you are" part.

## UNCHANGED: _create_session()

**File:** `auth/routes.py` — this function is **character-for-character identical** to Phase 1

```python
def _create_session(user, response, db):
    token = secrets.token_hex(32)

    session = SessionRecord(
        session_token=token,
        user_id=user.id,
        expires_at=datetime.now(timezone.utc) + timedelta(hours=24),
    )
    db.add(session)
    db.commit()

    response.set_cookie(
        key="session_token",
        value=token,
        httponly=True,
        samesite="lax",
        max_age=86400,
    )
```

**Nothing changed.** This is the whole point of the workshop.

# Code Walkthrough: What Changed in the Frontend

## REMOVED

The entire email form — input fields, signup/login toggle, error handling.

## ADDED

One button. That's it.

**File:** `frontend/src/App.jsx` → `LoginPage`

```jsx
function LoginPage({ onBack }) {

  const handleGoogleLogin = () => {
    // Redirect to our backend, which redirects to Google
    window.location.href = "http://localhost:8000/auth/google/login";
  };

  return (
    <div className="login-card">
      <h2>Welcome to AgentFlow</h2>
      <p>Sign in with your Google account to continue</p>

      <button className="login-google" onClick={handleGoogleLogin}>
        <GoogleIcon />
        Continue with Google
      </button>
    </div>
  );
}
```

**What happens when you click the button:**
1. Browser goes to `localhost:8000/auth/google/login`
2. Server redirects browser to Google
3. User logs in at Google
4. Google redirects to `localhost:8000/auth/callback`
5. Server creates session, sets cookie, redirects to `localhost:5173`
6. Frontend calls `/auth/me`, gets user data, shows chat page

The frontend never handles tokens, codes, or secrets. It just redirects and reads cookies.

# Security: Why This Flow is Secure

## 1. Your app never sees the user's Google password
Google handles the login form entirely. The password goes to Google's servers, not yours.

## 2. The authorization code is useless alone
Google gives your server a one-time `code`. But to exchange it for tokens, you need the `client_secret`. The `client_secret` lives in your `.env` file on the server. It never goes to the browser.

```
Browser sees:    code=ABCDEF  (useless without secret)
Server has:      code=ABCDEF + client_secret=GOCSPX-xyz  (can exchange for tokens)
```

## 3. The state parameter prevents CSRF
Before redirecting to Google, we generate a random `state` and store it. When Google redirects back, we check if the `state` matches. An attacker can't forge this.

```python
# Before redirect:
state = secrets.token_hex(16)     # "a1b2c3d4..."
pending_states[state] = True       # remember it

# After Google redirects back:
if state not in pending_states:    # check it
    raise "CSRF attack!"
del pending_states[state]          # one-time use
```

## 4. Tokens stay server-side
The `access_token` from Google is used server-to-server to get user info. It never reaches the browser. The browser only gets a session cookie (same as Phase 1).

In [ ]:
# Let's prove what DIDN'T change by comparing Phase 1 and Phase 2

phase1_session = """
def _create_session(user, response, db):
    token = secrets.token_hex(32)
    session = SessionRecord(
        session_token=token,
        user_id=user.id,
        expires_at=datetime.now(timezone.utc) + timedelta(hours=24),
    )
    db.add(session)
    db.commit()
    response.set_cookie(
        key="session_token",
        value=token,
        httponly=True,
        samesite="lax",
        max_age=86400,
    )
"""

phase2_session = """
def _create_session(user, response, db):
    token = secrets.token_hex(32)
    session = SessionRecord(
        session_token=token,
        user_id=user.id,
        expires_at=datetime.now(timezone.utc) + timedelta(hours=24),
    )
    db.add(session)
    db.commit()
    response.set_cookie(
        key="session_token",
        value=token,
        httponly=True,
        samesite="lax",
        max_age=86400,
    )
"""

if phase1_session == phase2_session:
    print("_create_session() is IDENTICAL in both phases")
    print("")
    print("Phase 1 email login and Phase 2 Google login")
    print("both call the exact same session creation code.")
    print("")
    print("The ONLY thing that changed is how we get the user's email:")
    print("  Phase 1: user typed it (unverified)")
    print("  Phase 2: Google confirmed it (verified)")
else:
    print("Something changed! Check the code.")

# Live Demo: Test Google Login

## Start servers

```bash
# Terminal 1 — Backend
cd agent-flow
python -m uvicorn app:app --reload --port 8000

# Terminal 2 — Frontend
cd agent-flow/frontend
npm run dev
```

## Test flow

1. Open `http://localhost:5173`
2. Click **"Start chatting"**
3. Click **"Continue with Google"**
4. Google login page appears — sign in with your Google account
5. You're redirected back to AgentFlow
6. Your **real name and profile picture** appear in the nav (not a typed name like Phase 1)
7. Send a chat message — works
8. Open DevTools → Application → Cookies → `session_token` is there (same as Phase 1)
9. Click **"Log out"** → cookie gone, back to landing page

## What to point out to the class

**In DevTools → Network tab during login, you'll see these redirects:**

```
1. GET localhost:8000/auth/google/login         → 302 redirect
2. GET accounts.google.com/o/oauth2/v2/auth...  → Google login page
3. GET localhost:8000/auth/callback?code=...    → 302 redirect
4. GET localhost:5173                           → frontend loads
5. GET localhost:8000/auth/me                   → returns user with real name + picture
```

**Compare with Phase 1:**
- Phase 1: POST /auth/signup + POST /auth/login → cookie set
- Phase 2: browser redirects through Google → same cookie set

The cookie in the browser looks **exactly the same** in both phases. Same `session_token`, same HttpOnly flag. Students can verify this.

# Summary: Phase 1 vs Phase 2

| | Phase 1 (email) | Phase 2 (Google) |
|---|---|---|
| **How user identifies** | Types email in a form | Clicks "Login with Google" |
| **Who verifies identity** | Nobody (we trust the email) | Google (OAuth2/OIDC) |
| **User creation** | Manual signup endpoint | Automatic (Just-In-Time on first login) |
| **Password handling** | No password at all | Google handles it (we never see it) |
| **Session creation** | `_create_session()` | Same `_create_session()` |
| **Cookie mechanism** | `set_cookie(httponly=True)` | Same `set_cookie(httponly=True)` |
| **Session lookup (/me)** | Read cookie → query DB | Same read cookie → query DB |
| **Logout** | Delete session + clear cookie | Same delete session + clear cookie |
| **Gatekeeper (Depends)** | `get_current_user()` | Same `get_current_user()` |
| **Profile data** | Whatever user typed | Real name + picture from Google |

**Bottom line:** 4 things changed (identity flow). 5 things stayed the same (session infrastructure).

The session layer is a reusable foundation. You can swap the identity provider (Google, GitHub, Microsoft, SAML) without touching sessions, cookies, logout, or route protection.